In [2]:
!wget https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
!unzip trainingandtestdata.zip

--2026-06-22 14:54:21--  https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81363704 (78M) [application/zip]
Saving to: ‘trainingandtestdata.zip’

trainingandtestdata 100%[===================>]  77.59M  89.4MB/s    in 0.9s    

2026-06-22 14:54:22 (89.4 MB/s) - ‘trainingandtestdata.zip’ saved [81363704/81363704]

Archive:  trainingandtestdata.zip
  inflating: testdata.manual.2009.06.14.csv  
  inflating: training.1600000.processed.noemoticon.csv  


In [3]:
import pandas as pd

# Training set
train_df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None,
    names=["label", "id", "date", "query", "user", "text"]
)

# Test set
test_df = pd.read_csv(
    "testdata.manual.2009.06.14.csv",
    encoding="latin-1",
    header=None,
    names=["label", "id", "date", "query", "user", "text"]
)

# Convert labels: 0 -> 0, 4 -> 1
train_df["label"] = train_df["label"].map({0: 0, 4: 1})
test_df["label"] = test_df["label"].map({0: 0, 4: 1})

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (1600000, 6)
Test shape: (498, 6)


,label,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [4]:
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [5]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags (#happy -> happy)
    text = re.sub(r'#', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]

    return " ".join(words)

In [6]:
columns = ["label", "id", "date", "query", "user", "text"]

df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None,
    names=columns
)

# Convert labels: 0 -> 0, 4 -> 1
df["label"] = df["label"].map({0: 0, 4: 1})

In [7]:
df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text"]].head()

,text,clean_text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",thats bummer shoulda got david carr third day
1,is upset that he can't update his Facebook by ...,upset cant update facebook texting might cry r...
2,@Kenichan I dived many times for the ball. Man...,dived many time ball managed save rest go bound
3,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,"@nationwideclass no, it's not behaving at all....",behaving im mad cant see


In [8]:
from sklearn.model_selection import train_test_split

sample_df = df.sample(n=200000, random_state=42)

X = sample_df["clean_text"]
y = sample_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=50000,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(150000,)
(50000,)


In [9]:
import time
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [10]:
# =========================
# 4. Evaluation Function
# =========================

def evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    # Training time
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    # Inference time
    start_infer = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time() - start_infer

    inference_per_example_ms = (inference_time / len(X_test)) * 1000

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Save model and get size
    filename = model_name.replace(" ", "_").lower() + ".joblib"
    joblib.dump(model, filename)
    model_size_mb = os.path.getsize(filename) / (1024 * 1024)

    print(f"\n===== {model_name} =====")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("Training time:", train_time, "seconds")
    print("Inference time per example:", inference_per_example_ms, "ms")
    print("Model size:", model_size_mb, "MB")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "Training Time (s)": train_time,
        "Inference (ms/example)": inference_per_example_ms,
        "Model Size (MB)": model_size_mb
    }

In [11]:
# =========================
# 5. Model 1: TF-IDF + Logistic Regression
# =========================

logistic_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="liblinear"
    ))
])

logistic_results = evaluate_model(
    logistic_model,
    "TF-IDF Logistic Regression",
    X_train,
    y_train,
    X_test,
    y_test
)


===== TF-IDF Logistic Regression =====
Accuracy: 0.76648
F1 Score: 0.7707531610775151
Training time: 6.580058813095093 seconds
Inference time per example: 0.013476734161376952 ms
Model size: 2.2436981201171875 MB

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.75      0.76     24964
           1       0.76      0.78      0.77     25036

    accuracy                           0.77     50000
   macro avg       0.77      0.77      0.77     50000
weighted avg       0.77      0.77      0.77     50000



In [12]:
# =========================
# 6. Model 2: TF-IDF + Naive Bayes
# =========================

nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", MultinomialNB())
])

nb_results = evaluate_model(
    nb_model,
    "TF-IDF Naive Bayes",
    X_train,
    y_train,
    X_test,
    y_test
)


===== TF-IDF Naive Bayes =====
Accuracy: 0.75634
F1 Score: 0.7575860079192949
Training time: 4.4791035652160645 seconds
Inference time per example: 0.02088493824005127 ms
Model size: 3.388016700744629 MB

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.75      0.76     24964
           1       0.75      0.76      0.76     25036

    accuracy                           0.76     50000
   macro avg       0.76      0.76      0.76     50000
weighted avg       0.76      0.76      0.76     50000



In [13]:
# =========================
# 6. Model 2: TF-IDF + Naive Bayes
# =========================

nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", MultinomialNB())
])

nb_results = evaluate_model(
    nb_model,
    "TF-IDF Naive Bayes",
    X_train,
    y_train,
    X_test,
    y_test
)


===== TF-IDF Naive Bayes =====
Accuracy: 0.75634
F1 Score: 0.7575860079192949
Training time: 5.185410499572754 seconds
Inference time per example: 0.02052196502685547 ms
Model size: 3.388016700744629 MB

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.75      0.76     24964
           1       0.75      0.76      0.76     25036

    accuracy                           0.76     50000
   macro avg       0.76      0.76      0.76     50000
weighted avg       0.76      0.76      0.76     50000



In [14]:
# =========================
# 7. Comparison Table
# =========================

results_df = pd.DataFrame([
    logistic_results,
    nb_results
])

results_df

,Model,Accuracy,F1 Score,Training Time (s),Inference (ms/example),Model Size (MB)
0,TF-IDF Logistic Regression,0.76648,0.770753,6.580059,0.013477,2.243698
1,TF-IDF Naive Bayes,0.75634,0.757586,5.185410,0.020522,3.388017


In [1]:
def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.split()

In [16]:
# =========================
# 1. Imports
# =========================

import time
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [21]:
# =========================
# 2. Device
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [17]:
# =========================
# 4. Build Vocabulary
# =========================

MAX_VOCAB_SIZE = 30000
MIN_FREQ = 2

counter = Counter()

for text in X_train:
    counter.update(simple_tokenizer(text))

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, freq in counter.most_common(MAX_VOCAB_SIZE - 2):
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

Vocabulary size: 25950


In [18]:
# =========================
# 5. Encode Text
# =========================

MAX_LEN = 40

def encode_text(text):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]

    if len(ids) < MAX_LEN:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

In [19]:
# =========================
# 8. BiLSTM Model
# =========================

class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        output_dim=1,
        num_layers=1,
        dropout=0.3
    ):
        super(BiLSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)

        _, (hidden, _) = self.lstm(embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        hidden_cat = torch.cat((forward_hidden, backward_hidden), dim=1)

        output = self.dropout(hidden_cat)
        output = self.fc(output)

        return output.squeeze(1)

In [22]:
# =========================
# 9. Initialize Model
# =========================

model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

BiLSTMClassifier(
  (embedding): Embedding(25950, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [25]:
# =========================
# 6. Dataset Class
# =========================

class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode_text(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.float)
        return x, y

In [26]:
# =========================
# 7. DataLoaders
# =========================

BATCH_SIZE = 128

train_dataset = SentimentDataset(X_train, y_train)
test_dataset = SentimentDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [27]:
# =========================
# 10. Training Loop
# =========================

EPOCHS = 3

start_train = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for texts, labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {avg_loss:.4f}")

training_time = time.time() - start_train

print("Training time:", training_time, "seconds")

Epoch 1/3 | Loss: 0.5504
Epoch 2/3 | Loss: 0.4683
Epoch 3/3 | Loss: 0.4277
Training time: 32.181116580963135 seconds


In [28]:
# =========================
# 11. Evaluation
# =========================

model.eval()

all_preds = []
all_labels = []

start_infer = time.time()

with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)

        outputs = model(texts)
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).long().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

inference_time = time.time() - start_infer
inference_ms_per_example = (inference_time / len(test_dataset)) * 1000

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Inference time per example:", inference_ms_per_example, "ms")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

Accuracy: 0.77228
F1 Score: 0.7692151775579699
Inference time per example: 0.039234561920166014 ms

Classification Report:
              precision    recall  f1-score   support

         0.0       0.76      0.79      0.78     24964
         1.0       0.78      0.76      0.77     25036

    accuracy                           0.77     50000
   macro avg       0.77      0.77      0.77     50000
weighted avg       0.77      0.77      0.77     50000



In [33]:
# =========================
# 12. Model Size
# =========================

torch.save(model.state_dict(), "bilstm_sentiment_model.pt")

model_size_mb = os.path.getsize("bilstm_sentiment_model.pt") / (1024 * 1024)

print("Model size:", model_size_mb, "MB")

Model size: 10.801806449890137 MB


In [34]:
# =========================
# 13. Save Results
# =========================

bilstm_results = {
    "Model": "BiLSTM on Word Embeddings",
    "Accuracy": accuracy,
    "F1 Score": f1,
    "Training Time (s)": training_time,
    "Inference (ms/example)": inference_ms_per_example,
        "Model Size (MB)": model_size_mb
}

bilstm_results

{'Model': 'BiLSTM on Word Embeddings',
 'Accuracy': 0.77228,
 'F1 Score': 0.7692151775579699,
 'Training Time (s)': 32.181116580963135,
 'Inference (ms/example)': 0.039234561920166014,
 'Model Size (MB)': 10.801806449890137}

In [36]:
# =========================
# Compare Classic ML + BiLSTM
# =========================

results_df = pd.DataFrame([
    logistic_results,
    nb_results,
    bilstm_results
])

# Make table cleaner
results_df = results_df.round({
    "Accuracy": 4,
    "F1 Score": 4,
    "Training Time (s)": 2,
    "Inference (ms/example)": 4,
    "Model Size (MB)": 2
})

results_df

,Model,Accuracy,F1 Score,Training Time (s),Inference (ms/example),Model Size (MB)
0,TF-IDF Logistic Regression,0.7665,0.7708,6.58,0.0135,2.24
1,TF-IDF Naive Bayes,0.7563,0.7576,5.19,0.0205,3.39
2,BiLSTM on Word Embeddings,0.7723,0.7692,32.18,0.0392,10.80


In [37]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [38]:
model_name = "distilbert-base-uncased"

tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [44]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [56]:
# train_texts = list(X_train[:20000])
# train_labels = list(y_train[:20000])

# test_texts = list(X_test[:5000])
# test_labels = list(y_test[:5000])

train_texts = list(X_train)
train_labels = list(y_train)

test_texts = list(X_test)
test_labels = list(y_test)


train_dataset = SentimentDataset(
    train_texts,
    train_labels,
    tokenizer
)

test_dataset = SentimentDataset(
    test_texts,
    test_labels,
    tokenizer
)

In [39]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [41]:
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [50]:
import numpy as np

In [57]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
start_train = time.time()

trainer.train()

distilbert_training_time = time.time() - start_train

print("Training time:", distilbert_training_time, "seconds")

Epoch,Training Loss,Validation Loss


In [52]:
start_infer = time.time()

predictions = trainer.predict(test_dataset)

distilbert_inference_time = time.time() - start_infer
distilbert_inference_ms = (
    distilbert_inference_time / len(test_dataset)
) * 1000

logits = predictions.predictions
y_pred = np.argmax(logits, axis=1)
y_true = predictions.label_ids

distilbert_accuracy = accuracy_score(y_true, y_pred)
distilbert_f1 = f1_score(y_true, y_pred)

print("Accuracy:", distilbert_accuracy)
print("F1 Score:", distilbert_f1)
print("Inference time per example:", distilbert_inference_ms, "ms")

print(classification_report(y_true, y_pred))

Accuracy: 0.7694
F1 Score: 0.7785673132321874
Inference time per example: 1.4891117572784425 ms
              precision    recall  f1-score   support

           0       0.77      0.75      0.76      2437
           1       0.77      0.79      0.78      2563

    accuracy                           0.77      5000
   macro avg       0.77      0.77      0.77      5000
weighted avg       0.77      0.77      0.77      5000



In [53]:
trainer.save_model("./distilbert_sentiment_model")

distilbert_model_size = 0

for path, dirs, files in os.walk("./distilbert_sentiment_model"):
    for file in files:
        file_path = os.path.join(path, file)
        distilbert_model_size += os.path.getsize(file_path)

distilbert_model_size_mb = distilbert_model_size / (1024 * 1024)

print("Model size:", distilbert_model_size_mb, "MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model size: 255.43062686920166 MB


In [54]:
distilbert_results = {
    "Model": "DistilBERT Fine-tune",
    "Accuracy": distilbert_accuracy,
    "F1 Score": distilbert_f1,
    "Training Time (s)": distilbert_training_time,
    "Inference (ms/example)": distilbert_inference_ms,
    "Model Size (MB)": distilbert_model_size_mb
}

In [55]:
results_df = pd.DataFrame([
    logistic_results,
    nb_results,
    bilstm_results,
    distilbert_results
])

results_df = results_df.round({
    "Accuracy": 4,
    "F1 Score": 4,
    "Training Time (s)": 2,
    "Inference (ms/example)": 4,
    "Model Size (MB)": 2
})

results_df

,Model,Accuracy,F1 Score,Training Time (s),Inference (ms/example),Model Size (MB)
0,TF-IDF Logistic Regression,0.7665,0.7708,6.58,0.0135,2.24
1,TF-IDF Naive Bayes,0.7563,0.7576,5.19,0.0205,3.39
2,BiLSTM on Word Embeddings,0.7723,0.7692,32.18,0.0392,10.80
3,DistilBERT Fine-tune,0.7694,0.7786,235.43,1.4891,255.43
